# Download of CMIP6 data

This notebook aims to fetch CMIP6 climate data from pangeo API and store as zarr files.
The scenarios / experiments are available in scenario_experiment_combination.json.
To be able to simulate cyclone tracks with a certain scenario / experiment, it will be necessary to download the historical data from the same scenario.

The data downloaded with be stored in data/input/cmip6_data, in a subfolder scenario/experiment.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import ipywidgets as widgets
from IPython.display import display

from src.data.cmip6_pangeo_regrid import load_scenario_queries, run_regridding_workflow

In [2]:
scenario_queries = load_scenario_queries('../scenario_experiment_combination.json')

models = sorted({query['source_id'] for query in scenario_queries})

source_dropdown = widgets.Dropdown(options=models, description='Model:')
experiment_dropdown = widgets.Dropdown(description='Experiment:')

def update_experiments(change):
    selected_source = change['new']
    matching_queries = [query for query in scenario_queries if query['source_id'] == selected_source]
    experiments = sorted({query['experiment_id'] for query in matching_queries})
    experiment_dropdown.options = ['-- Select an experiment --', *experiments]
    experiment_dropdown.value = '-- Select an experiment --'

source_dropdown.observe(update_experiments, names='value')
update_experiments({'new': source_dropdown.value})
display(source_dropdown, experiment_dropdown)

Dropdown(description='Model:', options=('ACCESS-CM2', 'FGOALS-g3', 'IPSL-CM6A-LR', 'MIROC-ES2L', 'MPI-ESM1-2-L…

Dropdown(description='Experiment:', options=('-- Select an experiment --', 'historical', 'ssp245', 'ssp370', '…

Attention : à date, les scénarios MIROC et UKESM10L ne fonctionnent pas. 

In [ ]:
source_id = source_dropdown.value
experiment_id = experiment_dropdown.value

if experiment_id == '-- Select an experiment --':
    print('Please select a model and an experiment.')
else:
    selected_query = next(
        query for query in scenario_queries
        if query['source_id'] == source_id and query['experiment_id'] == experiment_id
    )
    print(f'Selected model: {source_id}, experiment: {experiment_id}')
    run_regridding_workflow(
        scenario_queries=[selected_query],
        store_dir='../data/input/cmip6_data',
    )